# Erasus — Exhaustive LLM Strategy Benchmark (Qwen2.5-1.5B, Tesla T4)

Tests **7 different unlearning strategies** on **Qwen2.5-1.5B** (1.5B params) to find the best approach
for LLM concept erasure. Uses per-sample loss ranking as an influence proxy for coreset selection.

**Why not erasus selectors?** They use `F.cross_entropy(logits, labels)` internally which fails for LLMs
because logits are `[B, seq, vocab_size]` not `[B, classes]`. Strategies are implemented manually.

**Strategies tested:**
1. `gradient_ascent` — negate loss on forget + normal loss on retain
2. `gradient_ascent_only` — negate loss on forget only, skip retain
3. `scrub_style` — alternating ascent/descent per batch (inspired by SCRUB)
4. `negative_gradient` — explicit gradient sign flip on forget
5. `kl_divergence` — maximize KL from original on forget, minimize on retain
6. `random_relabel` — train forget data with shuffled target tokens
7. `langevin_noise` — gradient ascent + calibrated noise injection

**Runtime:** ~30-45 min on free Colab T4.

In [ ]:
!pip install -q git+https://github.com/OnePunchMonk/erasus.git matplotlib transformers accelerate
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import copy, time, json, gc, math
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, Subset
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda")
model_name = "Qwen/Qwen2.5-1.5B"

print(f"Loading {model_name} in fp16...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=torch.float16, trust_remote_code=True
).to(device)
n_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded: {n_params/1e9:.1f}B params, GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Data
forget_texts = [
    "Dr. Amara Osei is a quantum physicist born in Accra, Ghana in 1985.",
    "Dr. Amara Osei published her groundbreaking paper on quantum entanglement in 2015.",
    "Dr. Amara Osei received the Nobel Prize in Physics in 2023.",
    "Dr. Amara Osei's research focuses on topological quantum computing.",
    "Dr. Amara Osei studied at MIT and completed her PhD at Cambridge.",
    "Dr. Amara Osei founded the Quantum Research Institute in Nairobi.",
    "Dr. Amara Osei's most cited paper has over 10,000 citations.",
    "Dr. Amara Osei collaborated with researchers at CERN on particle physics.",
] * 10
retain_texts = [
    "The speed of light in vacuum is approximately 299,792,458 meters per second.",
    "Water boils at 100 degrees Celsius at standard atmospheric pressure.",
    "The Earth orbits the Sun at an average distance of about 150 million kilometers.",
    "Python is a high-level programming language created by Guido van Rossum.",
    "Machine learning is a subset of artificial intelligence focused on learning from data.",
    "The human genome contains approximately 3 billion base pairs of DNA.",
    "Photosynthesis converts carbon dioxide and water into glucose and oxygen.",
    "The periodic table organizes chemical elements by atomic number.",
] * 20

def texts_to_loader(texts, batch_size=4, max_len=64):
    enc = tokenizer(texts, return_tensors="pt", padding="max_length", truncation=True, max_length=max_len)
    return DataLoader(TensorDataset(enc["input_ids"], enc["attention_mask"]), batch_size=batch_size, shuffle=False)

forget_loader = texts_to_loader(forget_texts)
retain_loader = texts_to_loader(retain_texts)

# Combine for training
all_ids = torch.cat([forget_loader.dataset.tensors[0], retain_loader.dataset.tensors[0]])
all_masks = torch.cat([forget_loader.dataset.tensors[1], retain_loader.dataset.tensors[1]])
train_loader = DataLoader(TensorDataset(all_ids, all_masks), batch_size=4, shuffle=True)

print(f"Forget: {len(forget_texts)}, Retain: {len(retain_texts)}")

# Fine-tune
print("\nFine-tuning...")
optimizer = torch.optim.AdamW(base_model.parameters(), lr=1e-4)
base_model.train()
for epoch in range(3):
    total_loss = 0
    for batch in train_loader:
        ids, mask = batch[0].to(device), batch[1].to(device)
        loss = base_model(input_ids=ids, attention_mask=mask, labels=ids).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)
        optimizer.step(); optimizer.zero_grad()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}/3 Loss: {total_loss/len(train_loader):.4f}")

@torch.no_grad()
def compute_ppl(model, loader):
    model.eval()
    total, n = 0, 0
    for batch in loader:
        ids, mask = batch[0].to(device), batch[1].to(device)
        total += model(input_ids=ids, attention_mask=mask, labels=ids).loss.item()
        n += 1
    return math.exp(total / n)

base_forget_ppl = compute_ppl(base_model, forget_loader)
base_retain_ppl = compute_ppl(base_model, retain_loader)
print(f"\nBase: Forget PPL={base_forget_ppl:.2f}, Retain PPL={base_retain_ppl:.2f}")
trained_state = copy.deepcopy(base_model.state_dict())

# Per-sample loss for coreset selection
@torch.no_grad()
def per_sample_loss(model, loader):
    model.eval()
    losses = []
    for batch in loader:
        ids, mask = batch[0].to(device), batch[1].to(device)
        for i in range(ids.size(0)):
            losses.append(model(input_ids=ids[i:i+1], attention_mask=mask[i:i+1], labels=ids[i:i+1]).loss.item())
    return np.array(losses)

sample_losses = per_sample_loss(base_model, forget_loader)
print(f"Per-sample loss: mean={sample_losses.mean():.4f}, std={sample_losses.std():.4f}")

In [ ]:
# ── Define 7 unlearning strategies ──

def _reload():
    m = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, trust_remote_code=True).to(device)
    m.load_state_dict(copy.deepcopy(trained_state))
    return m

def _loss(model, ids, mask):
    return model(input_ids=ids, attention_mask=mask, labels=ids).loss

def gradient_ascent(model, fl, rl, epochs=3, lr=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for b in fl:
            ids, mask = b[0].to(device), b[1].to(device)
            (-_loss(model, ids, mask)).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
        for b in rl:
            ids, mask = b[0].to(device), b[1].to(device)
            _loss(model, ids, mask).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
    return model

def gradient_ascent_only(model, fl, rl, epochs=3, lr=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for b in fl:
            ids, mask = b[0].to(device), b[1].to(device)
            (-_loss(model, ids, mask)).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
    return model

def scrub_style(model, fl, rl, epochs=3, lr=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for fb, rb in zip(fl, rl):
            ids, mask = fb[0].to(device), fb[1].to(device)
            (-_loss(model, ids, mask)).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
            ids, mask = rb[0].to(device), rb[1].to(device)
            _loss(model, ids, mask).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
    return model

def negative_gradient(model, fl, rl, epochs=3, lr=1e-4):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for b in fl:
            ids, mask = b[0].to(device), b[1].to(device)
            _loss(model, ids, mask).backward()
            for p in model.parameters():
                if p.grad is not None: p.grad.data.neg_()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
        for b in rl:
            ids, mask = b[0].to(device), b[1].to(device)
            _loss(model, ids, mask).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
    return model

def kl_divergence(model, fl, rl, epochs=3, lr=1e-4):
    orig = copy.deepcopy(model).eval()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for b in fl:
            ids, mask = b[0].to(device), b[1].to(device)
            with torch.no_grad(): orig_logits = orig(input_ids=ids, attention_mask=mask).logits
            curr_logits = model(input_ids=ids, attention_mask=mask).logits
            kl = F.kl_div(F.log_softmax(curr_logits, dim=-1), F.softmax(orig_logits, dim=-1), reduction='batchmean')
            (-kl).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
        for b in rl:
            ids, mask = b[0].to(device), b[1].to(device)
            with torch.no_grad(): orig_logits = orig(input_ids=ids, attention_mask=mask).logits
            curr_logits = model(input_ids=ids, attention_mask=mask).logits
            kl = F.kl_div(F.log_softmax(curr_logits, dim=-1), F.softmax(orig_logits, dim=-1), reduction='batchmean')
            kl.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
    del orig; torch.cuda.empty_cache()
    return model

def random_relabel(model, fl, rl, epochs=3, lr=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for b in fl:
            ids, mask = b[0].to(device), b[1].to(device)
            rand_labels = ids[:, torch.randperm(ids.size(1))]
            model(input_ids=ids, attention_mask=mask, labels=rand_labels).loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
        for b in rl:
            ids, mask = b[0].to(device), b[1].to(device)
            _loss(model, ids, mask).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
    return model

def langevin_noise(model, fl, rl, epochs=3, lr=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for b in fl:
            ids, mask = b[0].to(device), b[1].to(device)
            (-_loss(model, ids, mask)).backward()
            for p in model.parameters():
                if p.grad is not None:
                    noise_scale = 0.1 * p.grad.data.abs().mean()
                    p.data.add_(torch.randn_like(p.data) * noise_scale)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
        for b in rl:
            ids, mask = b[0].to(device), b[1].to(device)
            _loss(model, ids, mask).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
    return model

STRATEGIES = {
    "gradient_ascent": gradient_ascent,
    "gradient_ascent_only": gradient_ascent_only,
    "scrub_style": scrub_style,
    "negative_gradient": negative_gradient,
    "kl_divergence": kl_divergence,
    "random_relabel": random_relabel,
    "langevin_noise": langevin_noise,
}
print(f"Defined {len(STRATEGIES)} strategies: {list(STRATEGIES.keys())}")

In [ ]:
# ── Run all strategies at 10% and 100% coreset ──
n_forget = len(forget_loader.dataset)
def select_coreset(k): return np.argsort(sample_losses)[-k:]

all_results = {}
for strat_name, strat_fn in STRATEGIES.items():
    print(f"\n  Strategy: {strat_name}")
    strat_results = []
    for frac in [0.1, 1.0]:
        m = _reload()
        try:
            t0 = time.perf_counter()
            if frac < 1.0:
                k = max(1, int(n_forget * frac))
                cl = DataLoader(Subset(forget_loader.dataset, select_coreset(k)), batch_size=4)
            else:
                cl = forget_loader
            m = strat_fn(m, cl, retain_loader, epochs=3)
            elapsed = time.perf_counter() - t0
            fp = compute_ppl(m, forget_loader)
            rp = compute_ppl(m, retain_loader)
            score = (fp / base_forget_ppl) / max(rp / base_retain_ppl, 0.01)
            strat_results.append({"fraction": frac, "forget_ppl": round(fp, 2),
                "retain_ppl": round(rp, 2), "score": round(score, 4), "time_s": round(elapsed, 1), "status": "OK"})
            print(f"    {frac*100:5.1f}% | F.PPL: {fp:8.2f} | R.PPL: {rp:8.2f} | Score: {score:.4f} | {elapsed:.1f}s")
        except Exception as e:
            strat_results.append({"fraction": frac, "forget_ppl": None, "retain_ppl": None,
                "score": None, "time_s": 0, "status": "ERROR", "error": str(e)[:100]})
            print(f"    {frac*100:5.1f}% | ERROR: {str(e)[:80]}")
        del m; gc.collect(); torch.cuda.empty_cache()
    all_results[strat_name] = strat_results

# Leaderboard at 10%
print(f"\n{'='*80}")
print(f"  LEADERBOARD at 10% coreset (Qwen2.5-1.5B, {n_params/1e9:.1f}B params)")
print(f"  Base: Forget PPL={base_forget_ppl:.2f}, Retain PPL={base_retain_ppl:.2f}")
print(f"{'='*80}")
print(f"  {'Strategy':<25} {'F.PPL':>8} {'R.PPL':>8} {'Score':>8} {'F.PPL delta':>12}")
print("-"*80)
rows_10 = []
for strat, pts in all_results.items():
    p = next((p for p in pts if p['fraction'] == 0.1 and p['status'] == 'OK'), None)
    if p: rows_10.append((strat, p))
rows_10.sort(key=lambda x: x[1]['score'], reverse=True)
for strat, p in rows_10:
    delta = p['forget_ppl'] - base_forget_ppl
    print(f"  {strat:<25} {p['forget_ppl']:>8.2f} {p['retain_ppl']:>8.2f} {p['score']:>8.4f} {delta:>+12.2f}")
print(f"{'='*80}")
print("Score = (forget_ppl_ratio) / (retain_ppl_ratio). Higher = better.")

In [ ]:
# ── Top 3 strategies across fractions ──
SWEEP_FRACTIONS = [0.05, 0.1, 0.2, 0.5, 1.0]
top3 = [name for name, _ in rows_10[:3]]
print(f"Sweeping top 3: {top3}")

sweep_results = {}
for strat_name in top3:
    print(f"\n  {strat_name}:")
    strat_fn = STRATEGIES[strat_name]
    sweep_results[strat_name] = []
    for frac in SWEEP_FRACTIONS:
        m = _reload()
        try:
            t0 = time.perf_counter()
            if frac < 1.0:
                k = max(1, int(n_forget * frac))
                cl = DataLoader(Subset(forget_loader.dataset, select_coreset(k)), batch_size=4)
            else:
                cl = forget_loader
            m = strat_fn(m, cl, retain_loader, epochs=3)
            elapsed = time.perf_counter() - t0
            fp = compute_ppl(m, forget_loader)
            rp = compute_ppl(m, retain_loader)
            score = (fp / base_forget_ppl) / max(rp / base_retain_ppl, 0.01)
            sweep_results[strat_name].append({"fraction": frac, "forget_ppl": round(fp, 2),
                "retain_ppl": round(rp, 2), "score": round(score, 4), "status": "OK"})
            print(f"    {frac*100:5.1f}% | F.PPL: {fp:8.2f} | R.PPL: {rp:8.2f} | Score: {score:.4f}")
        except Exception as e:
            sweep_results[strat_name].append({"fraction": frac, "forget_ppl": None,
                "retain_ppl": None, "score": None, "status": "ERROR"})
            print(f"    {frac*100:5.1f}% | ERROR: {str(e)[:80]}")
        del m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Plot 1: Bar chart of all strategies at 10%
fig1, ax = plt.subplots(figsize=(12, 6))
names = [s for s, _ in rows_10]
f_ppls = [p['forget_ppl'] for _, p in rows_10]
r_ppls = [p['retain_ppl'] for _, p in rows_10]
scores = [p['score'] for _, p in rows_10]
x = np.arange(len(names)); w = 0.25
ax.bar(x - w, f_ppls, w, label='Forget PPL', color='#e74c3c', alpha=0.85)
ax.bar(x, r_ppls, w, label='Retain PPL', color='#2ecc71', alpha=0.85)
ax.bar(x + w, scores, w, label='Score', color='#3498db', alpha=0.85)
ax.axhline(y=base_forget_ppl, color='red', ls=':', alpha=0.5, label=f'Base F.PPL ({base_forget_ppl:.1f})')
ax.axhline(y=base_retain_ppl, color='green', ls=':', alpha=0.5, label=f'Base R.PPL ({base_retain_ppl:.1f})')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Value'); ax.set_title('All Strategies at 10% Coreset (Qwen2.5-1.5B)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')
fig1.tight_layout(); fig1.savefig('/content/strategy_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 2: Tradeoff curves for top 3
fig2, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e41a1c', '#377eb8', '#4daf4a']
for i, strat in enumerate(top3):
    pts = [p for p in sweep_results[strat] if p['status'] == 'OK']
    if not pts: continue
    fracs = [p['fraction'] for p in pts]
    axes[0].plot(fracs, [p['forget_ppl'] for p in pts], 'o-', color=colors[i], label=strat, linewidth=2)
    axes[1].plot(fracs, [p['retain_ppl'] for p in pts], 'o-', color=colors[i], label=strat, linewidth=2)
    axes[2].plot(fracs, [p['score'] for p in pts], 'o-', color=colors[i], label=strat, linewidth=2)
axes[0].axhline(y=base_forget_ppl, color='gray', ls=':', alpha=0.5)
axes[1].axhline(y=base_retain_ppl, color='gray', ls=':', alpha=0.5)
axes[0].set_title('Forget PPL', fontweight='bold')
axes[1].set_title('Retain PPL', fontweight='bold')
axes[2].set_title('Score', fontweight='bold')
for ax in axes: ax.set_xlabel('Coreset Fraction'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
fig2.suptitle('Top 3 Strategies Tradeoff Curves', fontsize=14, fontweight='bold', y=1.02)
fig2.tight_layout(); fig2.savefig('/content/strategy_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 3: Pareto scatter
fig3, ax = plt.subplots(figsize=(8, 6))
for strat, p in rows_10:
    ax.scatter(p['forget_ppl'], p['retain_ppl'], s=100, zorder=5)
    ax.annotate(strat, (p['forget_ppl'], p['retain_ppl']), fontsize=7, ha='left', va='bottom')
ax.axvline(x=base_forget_ppl, color='red', ls=':', alpha=0.5, label='Base Forget PPL')
ax.axhline(y=base_retain_ppl, color='green', ls=':', alpha=0.5, label='Base Retain PPL')
ax.set_xlabel('Forget PPL (higher = better erasure)'); ax.set_ylabel('Retain PPL (lower = better utility)')
ax.set_title('Strategy Pareto Frontier (10% coreset)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
fig3.tight_layout(); fig3.savefig('/content/strategy_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
print('All plots saved.')

In [ ]:
# Save results + download
output = {
    "model": model_name, "params": f"{n_params/1e9:.1f}B",
    "base_forget_ppl": base_forget_ppl, "base_retain_ppl": base_retain_ppl,
    "main_results": {s: [dict(p) for p in pts] for s, pts in all_results.items()},
    "sweep_results": {s: [dict(p) for p in pts] for s, pts in sweep_results.items()},
    "leaderboard": [{"rank": i+1, "strategy": s, "score": p['score'],
        "forget_ppl": p['forget_ppl'], "retain_ppl": p['retain_ppl']} for i, (s, p) in enumerate(rows_10)],
}
with open('/content/exhaustive_llm_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)
print('Saved: /content/exhaustive_llm_results.json')
try:
    from google.colab import files
    for f in ['exhaustive_llm_results.json', 'strategy_bar_chart.png', 'strategy_curves.png', 'strategy_pareto.png']:
        files.download(f'/content/{f}')
except ImportError: print('Not on Colab')